# CA Experiment 4: QPM→SLM Interface Richness Ablation

**Runtime:** Colab Pro **A100 GPU** for the four Battery-C condition runs (Qwen2.5-7B inference) + Anthropic API (Claude Sonnet 4.5 judge) for scoring.

This notebook tests four progressively richer QPM→SLM interface designs against the Experiment 3 baseline:

| Condition | Interface design | New QPM signal exposed |
|---|---|---|
| **A** | Marginals only (Exp 3 replication) | None — continuity check |
| **B** | Marginals + purity / ambivalence field | Scalar coherence |
| **C** | Coherence-conditional speech-act modifier | Explicit behavioural directive |
| **D** | Marginals + purity + bivariate co-activations | Entanglement structure |

The QPM and SLM stack are held constant across the four conditions — only the interface function (`qpm_to_structured_intent_*`) and per-condition system-prompt augmentation vary.

**Hypothesis H_interface (primary):** at least one enriched condition (B, C, D) produces mean PersonaScore significantly higher than Condition A on the same 30-script bank, with paired-t p < 0.05 and d_z ≥ 0.2.

**Pre-registered prediction:** Condition C wins.

Estimated cost: ~$15.55 in Sonnet 4.5 judge calls + Colab Pro compute (~10 h A100 across the four conditions).

## Cell 1: Mount Google Drive & Set API Key

Upload the `CA_Experiment_4` folder (containing this notebook, all `.py` modules, and the `.env` file from Exp 1) to your Google Drive at `MyDrive/CA_Experiment_4/`. The API key is loaded with this priority:

1. Colab Secrets `ANTHROPIC_API_KEY` or `CHA_EXPERIMENT_SONNET_KEY` (key icon, left sidebar)
2. `CA_Experiment_4/.env` (if uploaded)
3. `CA_Experiment_3/.env` / `CA_Experiment_2/.env` / `CA_Experiment_1/.env` fallbacks

The `.env` file is gitignored, so it is safe to keep checked into each experiment folder locally and uploaded with the project to Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_4'
EXP2_DIR    = '/content/drive/MyDrive/CA_Experiment_2'
EXP1_DIR    = '/content/drive/MyDrive/CA_Experiment_1'

assert os.path.exists(PROJECT_DIR), (
    f'Upload CA_Experiment_4 to Drive first! Not found at {PROJECT_DIR}'
)
assert os.path.exists(EXP2_DIR), (
    f'CA_Experiment_2 must also be on Drive (needed for LoRA-10K adapter + SCI assets). '
    f'Not found at {EXP2_DIR}'
)
assert os.path.exists(EXP1_DIR), (
    f'CA_Experiment_1 must also be on Drive (needed for the 30-script evaluation bank '
    f'at CA_Experiment_1/scripts/). Not found at {EXP1_DIR}'
)

# API key — Colab Secrets preferred, then .env file on Drive.
# Every CA_Experiment_N/ now ships its own .env (same key); Exp 1 stays as
# the last fallback in case a freshly-seeded folder is missing one.
ANTHROPIC_API_KEY = ''
for secret_name in ('ANTHROPIC_API_KEY', 'CHA_EXPERIMENT_SONNET_KEY'):
    try:
        ANTHROPIC_API_KEY = userdata.get(secret_name)
        if ANTHROPIC_API_KEY:
            print(f'API key loaded from Colab Secrets ({secret_name})')
            break
    except Exception:
        pass

if not ANTHROPIC_API_KEY:
    for env_path in [
        os.path.join(PROJECT_DIR, '.env'),
        os.path.join(EXP1_DIR, '.env'),
    ]:
        if os.path.exists(env_path):
            with open(env_path) as f:
                for line in f:
                    for key_prefix in ('ANTHROPIC_API_KEY=', 'CHA_EXPERIMENT_SONNET_KEY='):
                        if line.startswith(key_prefix):
                            ANTHROPIC_API_KEY = line.strip().split('=', 1)[1]
                            print(f'API key loaded from {env_path}')
                            break
                    if ANTHROPIC_API_KEY:
                        break
            if ANTHROPIC_API_KEY:
                break

assert ANTHROPIC_API_KEY, (
    'No API key found. Set ANTHROPIC_API_KEY via Colab Secrets, '
    'or place it in a .env file under CA_Experiment_4/ or CA_Experiment_1/.'
)
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY

print(f'Project dir : {PROJECT_DIR}')
print(f'Exp 2 dir   : {EXP2_DIR}  (LoRA-10K adapter + SCI assets)')
print(f'Exp 1 dir   : {EXP1_DIR}  (30-script eval bank)')
print(f'API key     : ...{ANTHROPIC_API_KEY[-8:]}')

## Cell 2: Install Python Dependencies

The four Exp 4 conditions all rely on the same dependency surface as Exp 3 Battery C — Qiskit Aer for QPM, the Anthropic SDK for judging, plus the GPU inference stack (Qwen + LoRA) installed in Cell 3.

In [ ]:
# Core deps (CPU; the GPU stack is installed in Cell 3)
!pip install -q 'numpy<2' scipy matplotlib \
                'qiskit==1.2.4' 'qiskit-aer==0.15.1' pylatexenc \
                anthropic python-dotenv vaderSentiment

## Cell 2b: QPM Circuit Visualization

Renders the full 12-qubit circuit for the psychotherapy profile with a neutral d-vector, so the structure of the experimental QPM is auditable before any Battery C runs. The circuit is byte-identical to Experiment 3 — same 11 trait qubits + 1 ancilla, same Ry init / intra-CNOT / inter-CRz / context Ry / Lindblad noise stages.

In [ ]:
import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import matplotlib.pyplot as plt
from IPython.display import display
from qpm import QPM, QUBIT_LABELS, N_TRAIT_QUBITS, N_TOTAL_QUBITS
import ca_assets as A

profile = A.PROFILES['psychotherapy']
qpm_vis = QPM(profile, n_shots=256)

d_neutral = [0.5, 0.5, 0.5, 0.5, 0.3]
circuit = qpm_vis._build_circuit([d_neutral])

print(f'Qubits : {N_TOTAL_QUBITS} ({N_TRAIT_QUBITS} trait + 1 ancilla)')
print(f'Depth  : {circuit.depth()}')
print(f'Gates  : {dict(circuit.count_ops())}')
print()
print('Circuit layers:')
print('  Stage 1 \u2014 Ry initialization      : 11 Ry gates (one per trait qubit)')
print('  Stage 2 \u2014 Intra-domain CNOT       : 5 CX gates  (within-domain correlations)')
print('  Stage 3 \u2014 Inter-domain CRz        : 8 CRz gates (cross-domain correlations, X-CRz-X for \u03c1<0)')
print('  Stage 4 \u2014 Context Ry (d-vector)   : 10 Ry gates (non-commutative context layer)')
print('  Stage 5 \u2014 Lindblad noise carriers : 12 id gates (noise attached by AerSimulator)')
print('  Stage 6 \u2014 Measurement             : 11 measure  (trait qubits only; ancilla q11 traced out)')

# Qiskit's circuit.draw('mpl', ...) returns a Figure not registered with pyplot,
# so plt.show() won't render it \u2014 use display(fig).
fig = circuit.draw('mpl', fold=55, style={'fontsize': 7.5})
fig.set_size_inches(22, 9)
fig.suptitle(
    'QPM Circuit \u2014 psychotherapy profile | d_neutral=[0.5,0.5,0.5,0.5,0.3]',
    fontsize=11, y=1.01,
)
display(fig)
plt.close(fig)

## Cell 2c: Measurement Histogram & Trait Marginals

Runs a quick 512-shot QPM sample on the psychotherapy profile and visualises:
1. **Top-20 bitstring histogram** — what raw measurement outcomes the QPM emits
2. **Trait marginals bar chart** — per-aspect marginal p̂_k (green > 0.6, orange in 0.4–0.6, red < 0.4)

Used as a calibration check: the marginals should reproduce the Exp 3 psychotherapy pattern (high A_com / A_pol / E_ent, low N_vol / N_wth).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit.visualization import plot_histogram
from qpm import QPM, QUBIT_LABELS
import ca_assets as A

profile = A.PROFILES['psychotherapy']
qpm_vis = QPM(profile, n_shots=512)
d_neutral = [0.5, 0.5, 0.5, 0.5, 0.3]

result = qpm_vis.run(d_neutral, n_shots=512)
counts    = result['counts']
marginals = result['marginals']
purity    = result['purity_approx']

print(f'Shots               : {result["n_shots"]}')
print(f'Distinct bitstrings : {len(counts)} / {2**11} possible')
print(f'Purity proxy        : {purity:.4f}  (0=pure, 0.5=maximally mixed)')

# Bitstring histogram (top-20)
top20 = dict(sorted(counts.items(), key=lambda x: -x[1])[:20])
fig1 = plot_histogram(
    top20,
    title='Top-20 bitstrings \u2014 psychotherapy, d_neutral (512 shots)',
    figsize=(14, 4),
    color='#2196F3',
    bar_labels=False,
)
fig1.axes[0].set_xlabel('Bitstring (q10\u2026q0, right=q0)', fontsize=9)
fig1.axes[0].set_ylabel('Counts', fontsize=9)
fig1.axes[0].tick_params(axis='x', labelsize=7, rotation=45)
display(fig1)
plt.close(fig1)

# Trait marginal bar chart
fig2, ax = plt.subplots(figsize=(12, 4))
labels = QUBIT_LABELS
values = [marginals[lbl] for lbl in labels]
colors = [
    '#4CAF50' if v > 0.6 else '#FF9800' if v > 0.4 else '#F44336'
    for v in values
]
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='midpoint 0.5')
ax.set_ylim(0, 1)
ax.set_ylabel('Marginal probability  p\u0302_k', fontsize=10)
ax.set_title(
    f'QPM trait marginals \u2014 psychotherapy profile, d_neutral | purity_approx={purity:.4f}',
    fontsize=10
)
ax.legend(fontsize=9)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f'{v:.2f}',
            ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nTrait marginals:')
for lbl, v in marginals.items():
    bar = '\u2588' * int(v * 30)
    print(f'  {lbl:8s}: {v:.3f}  {bar}')

## Cell 2d: Per-condition structured-intent compare

**This is the experimental variable.** For one shared QPM run (psychotherapy profile, neutral d-vector), build the structured-intent JSON for each of the four conditions and render them side-by-side. Audits *exactly* what each interface ships to the SLM before any costly Battery C run.

Note: at the neutral d-vector the psychotherapy ambivalence is in the moderate band (~0.41), so Condition C's directive does **not** fire — Condition C reduces to Condition A in this snapshot. That is the intended design: C only changes the prompt when the QPM signals a genuinely conflicted internal state.

In [ ]:
import json
from qpm import QPM
import ca_assets as A

profile = A.PROFILES['psychotherapy']
qpm_vis = QPM(profile, n_shots=1024)
d_neutral = [0.5, 0.5, 0.5, 0.5, 0.3]

res = qpm_vis.run(d_neutral)
ambivalence = res['purity_approx']
purity_proxy = round(1.0 - ambivalence, 4)
print(f'Shared QPM state: ambivalence = {ambivalence:.4f}  (purity_proxy = {purity_proxy:.4f})')
print(f'Condition C threshold band: low < 0.15, high > 0.45')
print(f'\u2192 Condition C directive fires this turn? '
      f'{"YES" if ambivalence > 0.45 or ambivalence < 0.15 else "NO"}')

intents = {}
for cond in 'ABCD':
    kwargs = {'condition': cond, 'marginals': res['marginals'], 'd_vector': d_neutral}
    if cond in 'BCD': kwargs['purity_proxy'] = purity_proxy
    if cond == 'D':   kwargs['counts'] = res['counts']
    intents[cond] = A.qpm_to_structured_intent(**kwargs)

for cond in 'ABCD':
    intent = intents[cond]
    prompt = A.build_condition_system_prompt(cond, intent)
    print(f'\n=== Condition {cond} \u2014 {A.CONDITION_DESCRIPTIONS[cond]} ===')
    print(f'  intent top-level keys      : {sorted(intent.keys())}')
    print(f'  cognitive_state            : {intent.get("cognitive_state", {})}')
    if cond == "D":
        print(f'  trait_coactivation         : {intent["trait_coactivation"]}')
    if cond == "C":
        modifier = intent['cognitive_state'].get('speech_act_modifier')
        print(f'  speech_act_modifier        : {modifier}')
        print(f'  c_directives appended      : {len(intent["cognitive_state"]["c_directives"])}')
    print(f'  system-prompt size (chars) : {len(prompt)}')

# Render the prompt-size delta as a small bar chart
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
sizes = [len(A.build_condition_system_prompt(c, intents[c])) for c in 'ABCD']
cols = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']
bars = ax.bar(list('ABCD'), sizes, color=cols, edgecolor='white')
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, s + 30, f'{s}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('System-prompt size (chars)')
ax.set_title('Per-condition SLM system-prompt size  \u2014 neutral d-vector')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(sizes) * 1.10)
plt.tight_layout()
plt.show()

## Cell 2e: Trait co-activation bars (Condition D)

For the shared QPM run from the previous cell, render Condition D's 8 CRz joint probabilities `P(qᵢ=1 ∧ qⱼ=1)`. The 8 pairs come from Table 7 of the v3 paper (Stability cluster, Plasticity cluster, cross-factor edges).

- **Negative-ρ pairs** (orange) should show low joint probabilities — the QPM is suppressing the unlikely co-activation.
- **Positive-ρ pairs** (blue) should show higher joint probabilities — the QPM is reinforcing the likely co-activation.

If this pattern is visible, Condition D's interface is conveying real entanglement structure to the SLM that Conditions A/B/C discard.

In [ ]:
import matplotlib.pyplot as plt
import ca_assets as A

intent_d = intents['D']  # from previous cell
coacts = intent_d['trait_coactivation']

# Build display rows in the CRZ_PAIRS order, carrying \u03c1 sign for colour coding
rows = []
for qi, qj, label, rho in A.CRZ_PAIRS:
    rows.append((label, coacts[label], rho))

fig, ax = plt.subplots(figsize=(11, 4.5))
x = list(range(len(rows)))
values = [r[1] for r in rows]
colors = ['#1f77b4' if r[2] > 0 else '#ff7f0e' for r in rows]
bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.8)
for bar, (_, v, _) in zip(bars, rows):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f'{v:.2f}',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{r[0]}\n(\u03c1={r[2]:+.2f})' for r in rows],
    fontsize=9, rotation=15, ha='right',
)
ax.set_ylabel('Joint probability  P(q\u1d62=1 \u2227 q\u2c7c=1)')
ax.set_ylim(0, max(values) * 1.20 if values else 1.0)
ax.set_title('Condition D \u2014 trait co-activation joints (8 CRz-coupled pairs, neutral d-vector)')
ax.axhline(0.15, color='grey', linestyle=':', alpha=0.5, label='low: < 0.15  (mutual suppression)')
ax.axhline(0.50, color='grey', linestyle='--', alpha=0.5, label='high: > 0.50  (mutually reinforcing)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print(f'  Mean negative-\u03c1 pair joint: '
      f'{sum(r[1] for r in rows if r[2] < 0) / sum(1 for r in rows if r[2] < 0):.3f}')
print(f'  Mean positive-\u03c1 pair joint: '
      f'{sum(r[1] for r in rows if r[2] > 0) / sum(1 for r in rows if r[2] > 0):.3f}')

## Cell 3: Verify GPU & LoRA-10K Adapter

The four Battery-C condition runs each require:
1. **A100 or L4 GPU** for Qwen2.5-7B-Instruct (4-bit NF4)
2. The Experiment 2 **LoRA-10K** adapter at `CA_Experiment_2/adapters/lora_10k/` (already present in Drive after Exp 2).

If the adapter is missing, re-run Cells 8–10 of the Experiment 2 Colab notebook to retrain it.

In [ ]:
!nvidia-smi | head -20
import os
adapter_path = os.path.join(EXP2_DIR, 'adapters', 'lora_10k')
print('\nLoRA-10K adapter present:', os.path.isdir(adapter_path))
print('  path:', adapter_path)

# Install GPU inference deps (only needed for Battery C condition runs)
!pip install -q transformers peft bitsandbytes accelerate

## Cell 4: Condition A — Marginals only (continuity check)

**Plan §4.1.** Replicates the Experiment 3 QPM arm with no interface change.

**Continuity gate:** Condition A's mean PersonaScore should fall within ±0.05 of the Experiment 3 QPM result (4.410). If it does not, pause and investigate before running Conditions B/C/D.

Resumable — completed scripts are skipped on re-run. Expected wall-clock per condition: ~2.5 hours on A100.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 experiment_runner.py --condition A

In [ ]:
# Inline summary of Condition A after completion
import json, statistics
from pathlib import Path
from collections import defaultdict

cond_dir = Path(PROJECT_DIR) / 'logs' / 'condition_a_psychotherapy'
scores, by_dim = [], defaultdict(list)
for path in sorted(cond_dir.glob('scores_condition_a_*.jsonl')):
    for line in path.open():
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if 'score' in rec:
            scores.append(rec['score'])
            by_dim[rec['dimension']].append(rec['score'])

print(f'Condition A \u2014 n={len(scores)}  mean={statistics.mean(scores):.4f}')
for d in ('T', 'E', 'C', 'S'):
    vals = by_dim.get(d, [])
    if vals:
        print(f'  dim {d}: n={len(vals):>3}  mean={statistics.mean(vals):.4f}')
dev = statistics.mean(scores) - 4.410
print(f'\nContinuity \u0394 vs Exp 3 QPM (4.410): {dev:+.4f}  '
      f'\u2192 {"OK" if abs(dev) <= 0.05 else "FAIL \u2014 investigate before proceeding"}')

## Cell 4b: Reliability check (κ_w ≥ 0.70)

**Plan §5.4.** Re-judge a 5% random sample of Condition A probes at temperature 0 with seed 42 and compute Cohen's quadratic-weighted κ against the original scores.

Pass threshold: κ_w ≥ 0.70. If it fails, pause and review the rubric before running Conditions B/C/D.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python3 experiment_runner.py --reliability

## Cell 5: Condition B — Marginals + purity / ambivalence field

**Plan §4.2.** Adds a `cognitive_state` block (ambivalence, ambivalence_label, purity_proxy) to the persona JSON and appends a `[PERSONALITY STATE GUIDANCE]` paragraph to the system prompt.

The QPM call and shot count are identical to Condition A; only the structured intent and system prompt change.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python3 experiment_runner.py --condition B

In [ ]:
# Inline summary of Condition B
import json, statistics
from pathlib import Path
from collections import defaultdict

cond_dir = Path(PROJECT_DIR) / 'logs' / 'condition_b_psychotherapy'
scores, by_dim = [], defaultdict(list)
for path in sorted(cond_dir.glob('scores_condition_b_*.jsonl')):
    for line in path.open():
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        if 'score' in rec:
            scores.append(rec['score'])
            by_dim[rec['dimension']].append(rec['score'])
print(f'Condition B \u2014 n={len(scores)}  mean={statistics.mean(scores):.4f}')
for d in ('T', 'E', 'C', 'S'):
    vals = by_dim.get(d, [])
    if vals: print(f'  dim {d}: mean={statistics.mean(vals):.4f}')

## Cell 6: Condition C — Coherence-conditional speech-act modifier

**Plan §4.3.** Modifies the speech-act / constraints rather than adding a new top-level JSON block. When QPM ambivalence > 0.45, an `__with_expressed_uncertainty` behavioural modifier and an explicit "let the ambivalence be audible" constraint are injected into a high-attention `<current_directive>` block. When ambivalence < 0.15, the corresponding `__grounded` directive fires. Otherwise the prompt is byte-identical to Condition A.

**Pre-registered prediction:** Condition C produces the highest mean PersonaScore because the coherence signal reaches the SLM through the highest-attention channel already in the interface.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python3 experiment_runner.py --condition C

In [ ]:
# Inline summary of Condition C — includes how often the modifier fired
import json, statistics
from pathlib import Path
from collections import defaultdict

cond_dir = Path(PROJECT_DIR) / 'logs' / 'condition_c_psychotherapy'
scores, by_dim, fires = [], defaultdict(list), 0
for path in sorted(cond_dir.glob('scores_condition_c_*.jsonl')):
    for line in path.open():
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        if 'score' in rec:
            scores.append(rec['score'])
            by_dim[rec['dimension']].append(rec['score'])
            if rec.get('c_modifier'): fires += 1
print(f'Condition C \u2014 n={len(scores)}  mean={statistics.mean(scores):.4f}')
print(f'  Speech-act modifier fired on {fires}/{len(scores)} probes '
      f'({100*fires/max(len(scores),1):.1f}%)')
for d in ('T', 'E', 'C', 'S'):
    vals = by_dim.get(d, [])
    if vals: print(f'  dim {d}: mean={statistics.mean(vals):.4f}')

## Cell 7: Condition D — Marginals + purity + bivariate co-activations

**Plan §4.4.** Adds the `cognitive_state` block from Condition B plus a `trait_coactivation` dict containing joint probabilities for the 8 CRz-coupled qubit pairs (Stability + Plasticity + cross-factor edges from Table 7 of the v3 paper).

A `[TRAIT COACTIVATION GUIDANCE]` paragraph is appended after the personality guidance instructing the SLM how to read the co-activation values.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python3 experiment_runner.py --condition D

In [ ]:
# Inline summary of Condition D
import json, statistics
from pathlib import Path
from collections import defaultdict

cond_dir = Path(PROJECT_DIR) / 'logs' / 'condition_d_psychotherapy'
scores, by_dim = [], defaultdict(list)
for path in sorted(cond_dir.glob('scores_condition_d_*.jsonl')):
    for line in path.open():
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        if 'score' in rec:
            scores.append(rec['score'])
            by_dim[rec['dimension']].append(rec['score'])
print(f'Condition D \u2014 n={len(scores)}  mean={statistics.mean(scores):.4f}')
for d in ('T', 'E', 'C', 'S'):
    vals = by_dim.get(d, [])
    if vals: print(f'  dim {d}: mean={statistics.mean(vals):.4f}')

## Cell 8: Analysis — All four conditions + Decision Rule Outcome

Runs `analyse_results.py` which computes:

- Per-condition mean PersonaScore + by-dimension and by-turn breakdowns
- Continuity check vs Experiment 3 QPM (4.410)
- Paired t-tests B vs A, C vs A, D vs A at the (script, turn, dimension) level
- H_interface, H_C_wins, H_capability, H_purity_episodic verdicts
- Effect-size ladder (Exp 3 H1/H2/H3 anchors + Exp 4 best condition)
- The pre-registered decision rule's prescription for the CA v3 paper update

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python3 analyse_results.py --conditions A,B,C,D

In [ ]:
# Print the decision rule outcome inline
import json
from pathlib import Path
summary = json.loads((Path(PROJECT_DIR) / 'results' / 'analysis_data.json').read_text())

print('=== Hypothesis verdicts ===')
for k, v in summary['hypotheses'].items():
    print(f'  {k}: {v}')

print(f"\nBest condition: {summary['best_condition']}  (d_z = {summary['best_cohens_d']})")

print('\n=== Decision rule outcome ===')
print(' ', summary['decision_rule_outcome'])

print('\n=== Paper update prescription ===')
print(' ', summary['paper_update'])

print('\n=== Effect-size ladder ===')
for row in summary['effect_size_ladder']:
    print(f"  {row['experiment']:<10s} {row['metric']:<50s} d = {row['d']}")

if 'c_firing_rate' in summary:
    fr = summary['c_firing_rate']
    print('\n=== Condition C firing rate ===')
    print(f"  total turns                       : {fr['total_turns']}")
    print(f"  modifier fired                    : {fr['total_fires']} ({fr['fire_rate']*100:.1f}%)")
    print(f"  with_expressed_uncertainty (high) : {fr['with_expressed_uncertainty']}")
    print(f"  grounded (low)                    : {fr['grounded']}")

## Cell 9: View Results

`analyse_results.py` saves five plots into `results/`:

1. **Turn series** — PersonaScore by probe turn × condition (look for SCI refresh steps at 15 / 30)
2. **Dimension bars** — by-dimension mean × condition (where did Condition X help?)
3. **Effect-size ladder** — Exp 3 H1/H2/H3 anchors + Exp 4 best condition on a log scale
4. **Ambivalence distribution** — per-turn ambivalence histogram with Condition C's 0.15 / 0.45 decision bands
5. **Condition C firing rate by script** — % turns per script where C's speech-act modifier fired (correlate with the per-script PersonaScore deltas)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for name in ('exp4_turn_series_psychotherapy.png',
             'exp4_dimension_bars_psychotherapy.png',
             'exp4_effect_size_ladder.png',
             'exp4_ambivalence_distribution.png',
             'exp4_condition_c_firing_rate.png'):
    path = Path(PROJECT_DIR) / 'results' / name
    if path.exists():
        print(f'--- {name} ---')
        display(Image(str(path)))
    else:
        print(f'  (missing: {name})')

---

## Recovery After Disconnect

1. Re-run **Cell 1** (mount Drive, set API key)
2. Re-run **Cell 2** (install core deps)
3. Re-run **Cell 3** (GPU + adapter check, install GPU deps)
4. Re-run only the condition cells that did **not** finish — each condition is resumable per (script, condition) because completed `scores_condition_<x>_NNN.jsonl` files end with a `turn=40` line and are detected by `_is_script_done()`.
5. Once all four conditions complete, re-run **Cell 8** (analysis) — it reads everything from `logs/` so it is idempotent.

### Useful flags
- `--scripts 1-10`     restrict a condition run to a script-id range
- `--scripts 81-88`    re-run only the adversarial scripts
- `--shots 4096`       boost QPM shot count if the H4 SNR check from Exp 3 looks shaky
- `--reliability`      re-judge a 5% Condition A sample at T=0 (κ_w gate)

### If the continuity check fails
If Condition A mean drifts > ±0.05 from the Exp 3 QPM result (4.410) **stop** — something in the SLM stack, prompt template, or judge state differs from Exp 3. Compare against `CA_Experiment_3/results/analysis_data.json` to localise the drift.